# Défi 1 — Survie à 2 mois avec une régression logistique

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/ai_for_medicine/blob/main/ia-medecine/notebooks/iam-s1-defi.ipynb)

*IA pour la médecine — séance 1.*

    On prédit si un patient de réanimation est **vivant ou décédé 2 mois**
    après son admission, avec la méthode du TP 1 : une régression
    logistique sur des variables standardisées.

    **Le défi** : [https://ml-arena.com/viewchallenge/199](https://ml-arena.com/viewchallenge/199). Vous y déposez le fichier
    `submission.csv` produit à la fin de ce notebook ; le score est
    l'**exactitude** sur des patients dont vous ne connaissez pas le
    devenir. Répondre toujours « vivant » donne environ 0,59 : c'est le
    score à battre.

    Complétez les cellules marquées `# TODO` en remplaçant les `...`.

## 1. Les données

Les patients viennent de l'étude **SUPPORT2** (*Study to Understand
Prognoses and Preferences for Outcomes and Risks of Treatments*),
une étude multicentrique américaine de patients hospitalisés en
réanimation. Chaque patient est décrit par 28 variables mesurées à
l'admission : âge, sexe, groupe diagnostique, signes vitaux (pression
artérielle moyenne `meanbp`, fréquence cardiaque `hrt`, température
`temp`…), biologie (créatinine `crea`, sodium `sod`, pH `ph`…) et
scores de gravité (`aps`, `scoma`…).

La cible `outcome` vaut `alive` (vivant) ou `dead` (décédé) à 2 mois.

In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/racousin/ai_for_medicine/main/ia-medecine/donnees/support2/"
X_train = pd.read_csv(URL + "X_train.csv")
y_train = pd.read_csv(URL + "y_train_survie.csv")
X_test = pd.read_csv(URL + "X_test.csv")

assert (X_train["patient_id"] == y_train["patient_id"]).all()
print(f"Entraînement : {len(X_train)} patients ; test : {len(X_test)} patients")
X_train.head()

### Exercice 1 — La cible

Quelle proportion de patients est vivante à 2 mois ? C'est
l'exactitude d'un modèle qui répond toujours `alive`.

In [ ]:
y = y_train["outcome"]
# TODO : proportion de chaque classe
...

## 2. Les variables numériques

La régression logistique ne prend que des nombres. Pour ce premier
défi, on garde les **variables numériques** (les variables texte comme
`sex` ou `dzgroup` seront encodées en séance 2). Plusieurs ont des
**valeurs manquantes** : un examen non fait.

In [ ]:
# TODO : les colonnes numériques de X_train (select_dtypes)
X = ...
print(X.columns.tolist())
X.isna().mean().sort_values(ascending=False).head(8)

## 3. Le modèle

Trois étapes, enchaînées par `make_pipeline` :

1. `SimpleImputer(strategy="median")` remplace chaque valeur manquante
   par la médiane de la variable ;
2. `StandardScaler()` standardise (moyenne 0, écart-type 1) ;
3. `LogisticRegression(max_iter=1000)` classe.

`fit` apprend les trois étapes sur les seules données qu'on lui donne
(médianes, moyennes, écarts-types, coefficients) ; `predict` les
applique telles quelles. Les données de validation ne servent donc
jamais à l'entraînement : pas de fuite.

### Exercice 2 — Validation honnête

Réservez 20 % des patients pour la validation (`stratify=y`,
`random_state=42`), entraînez le pipeline sur les 80 % restants et
mesurez l'exactitude en validation.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

modele = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                       LogisticRegression(max_iter=1000))
# TODO : entraîner sur (X_tr, y_tr), prédire X_val
...
y_val_pred = ...

print(f"Exactitude en validation : {accuracy_score(y_val, y_val_pred):.3f}")
print(f"Toujours 'alive'         : {(y_val == 'alive').mean():.3f}")
ConfusionMatrixDisplay.from_predictions(y_val, y_val_pred, cmap="Blues")
plt.show()

Le modèle doit battre nettement la réponse constante. Regardez la
matrice : combien de décès sont prédits « vivant » ?

## 4. La soumission

Le choix est fait : on réentraîne sur **tous** les patients
d'entraînement, puis on prédit ceux du test.

### Exercice 3 — Réentraîner et prédire le test

In [ ]:
# TODO : entraîner modele sur (X, y), puis prédire les colonnes de X.columns dans X_test
...
predictions = ...

submission = pd.DataFrame({"patient_id": X_test["patient_id"],
                           "outcome": predictions})
assert len(submission) == 1541 and submission["patient_id"].is_unique
assert set(submission["outcome"]) <= {"alive", "dead"}
submission.to_csv("submission.csv", index=False)
print(f"submission.csv : {len(submission)} lignes")
submission["outcome"].value_counts()

Sur Colab, la cellule suivante télécharge `submission.csv` ; déposez-le
ensuite sur la page du défi : [https://ml-arena.com/viewchallenge/199](https://ml-arena.com/viewchallenge/199).

In [ ]:
try:
    from google.colab import files
except ImportError:
    print("Hors de Colab : submission.csv est dans le dossier courant.")
else:
    files.download("submission.csv")

## 5. Pour aller plus loin

- Le score du classement est-il proche de votre exactitude en
  validation ? Pourquoi peut-il différer un peu ?
- Remplacez la découpe unique par une validation croisée à 5 blocs
  (`cross_val_score(modele, X, y, cv=5)`) : quelle est la variabilité ?
- Calculez le **rappel** de la classe `dead` en validation : quelle part
  des décès le modèle annonce-t-il ?